# Image-side CLIP age probe (exploratory)
Plan: `docs/IMAGE_AGE_PROBE_PLAN.md`. **Not part of the preregistered study.**

For each face, CLIP's image embedding is scored against 80 anchors "a photo of a {N}-year-old person" (no race word) to give an implied age; the Black − White signed age error for minors is then computed with the audit's unchanged H2b code, as for MiVOLO and the FairFace classifier.

- **Step 2 (main result, CPU is enough):** ViT-L/14 on UTKFace, using the course project's stored embeddings (sha256-checked). No image is re-read.
- **Steps 3–4 (optional, T4, ~10 min each):** ViT-B/32 on UTKFace and ViT-L/14 on FairFace; embeddings are extracted with D4's own loader.

Every run first recomputes D4's margin from the embeddings and must reproduce the Phase 1 per-image scores (max difference < 0.001, identical decisions), or it stops.

**Ethics:** no images are generated or displayed; per-image outputs stay on Drive; only aggregate JSON goes to the repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, shutil, subprocess
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/child-detection-audit')
REPO = pathlib.Path('/content/child-detection-skin-tone-audit')
URL = 'github.com/isaacmutuma/child-skintone-detection.git'
os.environ['CHILD_AUDIT_ENV'] = 'colab'
os.chdir('/content')
!rm -rf {REPO}
# Public repo: plain clone. Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar;
# a fine-grained token with read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
assert (REPO / 'scripts/image_age_probe.py').exists(), 'image_age_probe.py not on GitHub yet: commit and push first'
os.environ.pop('UTKFACE_DIR', None); os.environ.pop('FAIRFACE_DIR', None)   # full data, not dev
!pip -q install open_clip_torch==3.3.0
import open_clip, torch; print('open_clip', open_clip.__version__, '| torch', torch.__version__)
!nvidia-smi --query-gpu=name --format=csv,noheader || echo 'CPU runtime: fine for step 2; steps 3-4 need a T4'

## 1. Unpack the pinned data (labels and file paths; sha256-checked)

In [ ]:
!python scripts/unpack_data.py
def show(arch, ds):
    d = json.loads((REPO / f'results/image_age_probe/image_age_probe_{arch}_{ds}.json').read_text())
    assert d['git']['dirty'] is False, 'code changed after clone'
    assert d['crosscheck_vs_phase1']['pass'], 'embeddings do not reproduce Phase 1'
    g = d['primary']['contrast']['age_err_gap_b_minus_a']
    print(f"{arch} {ds}: Black - White minor age error {g['gap']:+.2f} yr {g['ci']}, p_perm {g['p_perm']:.4f}")
    print('  decision:', d['decision']['outcome'], '| TPR gap check:', d['tpr_gap_check'])
    return d

## 2. ViT-L/14 on UTKFace (main result)

In [ ]:
EMB = '/content/drive/MyDrive/robust_child_filtering_model/embeddings/utkface__clip_vitl14.npz'
EMB_SHA = '4a6f66c1ec7b888f87ada759ab6bec3bde1407a077ff80ba28b12d83803bb3c3'
!python scripts/image_age_probe.py --arch ViT-L-14 --dataset utkface --embeddings {EMB} --embeddings-sha256 {EMB_SHA}
L14 = show('ViT-L-14', 'utkface')

## 3. Optional: ViT-B/32 on UTKFace (T4, ~10 min)

In [ ]:
RUN_OPTIONAL = False   # set True on a T4 runtime
if RUN_OPTIONAL:
    !python scripts/image_age_probe.py --arch ViT-B-32 --dataset utkface
    B32 = show('ViT-B-32', 'utkface')

## 4. Optional: ViT-L/14 on FairFace (children under ten only; bin midpoints; T4, ~10 min)

In [ ]:
if RUN_OPTIONAL:
    !python scripts/image_age_probe.py --arch ViT-L-14 --dataset fairface_val
    FF = show('ViT-L-14', 'fairface_val')

## 5. Save the aggregate JSONs to Drive (and a zip for the repo)

In [ ]:
import zipfile, hashlib
out_dir = DRIVE_ROOT / 'results' / 'image_age_probe'; out_dir.mkdir(parents=True, exist_ok=True)
files = sorted((REPO / 'results/image_age_probe').glob('*.json'))
for f in files:
    assert json.loads(f.read_text())['git']['dirty'] is False
    shutil.copy(f, out_dir / f.name); print('saved', f.name)
zp = DRIVE_ROOT / 'image_age_probe_outputs.zip'
with zipfile.ZipFile(zp, 'w') as z:
    for f in files:
        z.write(f, f.relative_to(REPO))
print(zp, zp.stat().st_size, 'bytes, sha256', hashlib.sha256(zp.read_bytes()).hexdigest())